<a href="https://colab.research.google.com/github/Satya-Padala/24P31A42V7_DriveReady_OPC_2026/blob/main/Day-3/task3.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [10]:
!pip install -q langchain langchain-groq

In [11]:
import os
from google.colab import userdata
os.environ["GROQ_API_KEY"] = userdata.get("GROQ_API_KEY")   # LangChain reads it from here

from langchain_groq import ChatGroq
llm = ChatGroq(model="openai/gpt-oss-120b", temperature=0.3)

In [12]:
from langchain_core.prompts import ChatPromptTemplate, MessagesPlaceholder
from langchain_core.output_parsers import StrOutputParser

SYSTEM = """ROLE:   You are the friendly assistant for Santhi's pickle shop.
FACTS:  Chicken pickle Rs 1000. Prawn pickle Rs 900. Mango pickle Rs 800 each.
        we prepare it after we get an order;veg and non-veg pickles are available;
        for one order it takes 5-6 days.
RULES:  no deliveries on sunday;if you do not know ,say
        "I'll check with the owner."
FORMAT: Reply in a polite way in customer's language;answer should be under 50 words."""

prompt = ChatPromptTemplate.from_messages([
    ("system", SYSTEM),
    MessagesPlaceholder("history"),          # the chat so far goes here
    ("human", "{question}"),
])
chain = prompt | llm | StrOutputParser()

history = []
def ask(question: str) -> str:
    answer = chain.invoke({"history": history, "question": question})
    history.extend([("human", question), ("ai", answer)])
    return answer

print(ask("Hi, I'm Satya. Do you deliver order on any day?"))
print(ask("What's my name?"))

Hi Satya! We deliver on any day except Sunday. Once you place an order, it takes about 5‑6 days to prepare before delivery. Let us know what you’d like!
Your name is Satya.


In [ ]:
history

[('human', "Hi, I'm Satya. Do you deliver order on any day?"),
 ('ai',
  'Hi Satya! We deliver on any day except Sunday. Once you place an order, we’ll prepare it in 5‑6 days and arrange delivery on a non‑Sunday. Let us know what you’d like!'),
 ('human', "What's my name?"),
 ('ai', 'Your name is Satya.')]

In [13]:
import csv

tests = [
    ["question", "must_include"],
    ["What is the price of chicken pickle?", "1000"],
    ["How much does prawn pickle cost?", "900"],
    ["What is the price of mango pickle?", "800"],
    ["Do you sell vegetarian pickles?", "vegetarian"],
    ["Do you have non-vegetarian pickles?", "non-vegetarian"],
    ["How many days does preparation take?", "5-6 days"],
    ["Do you deliver on Sundays?", "Sunday"],
    ["When do you prepare the pickles?", "after receiving an order"],
    ["Do you deliver to Kakinada?", "check with the owner"],
    ["Are chicken and prawn pickles available?", "chicken"]
]

with open("tests.csv", "w", newline="") as f:
    writer = csv.writer(f)
    writer.writerows(tests)

print("tests.csv created successfully!")

tests.csv created successfully!


In [18]:
import csv

def run_tests(path: str = "tests.csv") -> None:
    passed = 0
    total = 0

    with open(path, newline="") as f:
        rows = list(csv.DictReader(f))

    print(f"{'STATUS':<8} | {'QUESTION':<40} | {'EXPECTED':<20} | RESPONSE")
    print("-" * 130)

    for row in rows:
        question = row["question"]
        expected = row["must_include"]

        answer = chain.invoke({
            "history": [],
            "question": question
        })

        ok = expected.lower() in answer.lower()

        status = "PASS" if ok else "FAIL"

        print(f"{status:<8} | {question:<40} | {expected:<20} | {answer}")

        total += 1
        passed += ok

    print("-" * 130)
    print(f"Result: {passed}/{total} passed")
    print(f"Pass rate: {passed/total*100:.1f}%")

run_tests()

STATUS   | QUESTION                                 | EXPECTED             | RESPONSE
----------------------------------------------------------------------------------------------------------------------------------
PASS     | What is the price of chicken pickle?     | 1000                 | The chicken pickle costs Rs 1000.
PASS     | How much does prawn pickle cost?         | 900                  | The prawn pickle is priced at **Rs 900**.
PASS     | What is the price of mango pickle?       | 800                  | The mango pickle is priced at **Rs 800 each**. Let us know if you’d like to place an order!
PASS     | Do you sell vegetarian pickles?          | vegetarian           | Yes! We offer vegetarian pickles such as our mango pickle (₹800 each) alongside our non‑veg options. Let us know what you’d like, and we’ll prepare it for you.
FAIL     | Do you have non-vegetarian pickles?      | non-vegetarian       | Yes, we offer non‑vegetarian pickles—Chicken pickle at Rs 1000 and Pra

In [19]:
llm = ChatGroq(
    model="openai/gpt-oss-120b",
    temperature=1.0
)

chain = prompt | llm | StrOutputParser()

print("RUN 1")
run_tests()

print("\nRUN 2")
run_tests()

RUN 1
STATUS   | QUESTION                                 | EXPECTED             | RESPONSE
----------------------------------------------------------------------------------------------------------------------------------
PASS     | What is the price of chicken pickle?     | 1000                 | The chicken pickle costs Rs 1000.
PASS     | How much does prawn pickle cost?         | 900                  | Prawn pickle costs ₹ 900.
PASS     | What is the price of mango pickle?       | 800                  | Mango pickle costs Rs 800 each.
PASS     | Do you sell vegetarian pickles?          | vegetarian           | Yes, we offer vegetarian pickles such as our mango pickle. Let us know what you’d like, and we’ll prepare it for you!
FAIL     | Do you have non-vegetarian pickles?      | non-vegetarian       | Yes! We offer non‑vegetarian pickles – Chicken pickle for Rs 1000 and Prawn pickle for Rs 900. Let us know your choice, and we’ll prepare it after your order (takes 5‑6 days, no deli

Calculate the total price of pickles based on quantity. ------>Plain code

Answer customer questions about pickle prices and varieties.---->One LLM call

Receive an order, collect customer details, calculate the price, check the delivery day, and save the order.----->Agent
